# SQL

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from IPython.display import display
from sql.queries import QUERIES
from utils import db_connection as db

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)


def show(key, rows=15):
    """Print the question title and SQL, then show the result table."""
    q = QUERIES[key]
    print(f"{key.upper()} [{q['level']}] {q['title']}")
    print("-" * 70)
    print(q["sql"], "\n")
    df = db.run_query(q["sql"])
    print(f"{len(df)} rows")
    display(df.head(rows))
    return df

In [2]:
df = show("q09")

Q09 [Intermediate] All-rounders with 1000+ runs and 50+ wickets
----------------------------------------------------------------------
SELECT p.full_name,
       c.runs    AS total_runs,
       c.wickets AS total_wickets,
       c.format
FROM player_career_stats c
JOIN players p ON p.player_id = c.player_id
WHERE p.playing_role = 'All-rounder'
  AND c.runs > 1000
  AND c.wickets > 50
ORDER BY c.runs DESC; 

41 rows


,full_name,total_runs,total_wickets,format
0,Kabir Mehta,7816,154,ODI
1,Hasitha Peiris,7400,135,Test
2,Jason Maynard,6700,116,Test
3,Tyrone Burrowes,6330,151,Test
4,Callan Henare,6079,130,Test
5,Thabo Naidoo,5902,138,Test
6,Riley Fraser,5822,135,ODI
7,Theo Pemberton,5774,167,ODI
8,Sipho Dlamini,5697,171,ODI
9,Callum Calloway,5624,125,ODI


In [3]:
df = show("q10")

Q10 [Intermediate] Last 20 completed matches
----------------------------------------------------------------------
SELECT m.match_desc,
       t1.team_name AS team1,
       t2.team_name AS team2,
       w.team_name  AS winning_team,
       m.victory_margin,
       m.victory_type,
       v.venue_name,
       m.match_date
FROM matches m
JOIN teams t1 ON t1.team_id = m.team1_id
JOIN teams t2 ON t2.team_id = m.team2_id
LEFT JOIN teams w  ON w.team_id = m.winner_id
LEFT JOIN venues v ON v.venue_id = m.venue_id
WHERE m.state = 'Complete'
ORDER BY m.match_date DESC, m.match_id DESC
LIMIT 20; 

20 rows


,match_desc,team1,team2,winning_team,victory_margin,victory_type,venue_name,match_date
0,6th Match,Romania Women,Sweden Women,Sweden Women,10.0,wickets,"Moara Vlasiei Cricket Ground, Bucharest",2026-10-17
1,5th Match,Romania Women,Austria Women,Austria Women,7.0,wickets,"Moara Vlasiei Cricket Ground, Bucharest",2026-10-17
2,Pool B,Boland,Mpumalanga Rhinos,Boland,39.0,runs,Boland Park,2026-10-17
3,1st Match,Zimbabwe,Bangladesh,Zimbabwe,99.0,runs,Sheikh Zayed Stadium,2026-10-17
4,138th Match,Canada,Nepal,Canada,24.0,runs,Al Amerat Cricket Ground (Ministry Turf 1),2026-10-17
5,15th Match,Hobart Hurricanes Women,Melbourne Renegades Women,Melbourne Renegades Women,8.0,wickets,Blacktown Olympic Park No 2 Oval,2026-10-17
6,14th Match,Adelaide Strikers Women,Sydney Sixers Women,Adelaide Strikers Women,38.0,runs,Blacktown International Sportspark,2026-10-17
7,13th Match,ACT Women,Melbourne Stars Women,Melbourne Stars Women,4.0,wickets,Blacktown Olympic Park No 2 Oval,2026-10-17
8,10th Match,South Australia,Queensland,South Australia,126.0,runs,Karen Rolton Oval,2026-10-17
9,5th T20I,India,West Indies,NaN,NaN,no result,M.Chinnaswamy Stadium,2026-10-17


In [4]:
df = show("q11")

Q11 [Intermediate] Runs by format and overall batting average (players in 2+ formats)
----------------------------------------------------------------------
SELECT p.full_name,
       SUM(CASE WHEN c.format = 'Test' THEN c.runs END) AS test_runs,
       SUM(CASE WHEN c.format = 'ODI'  THEN c.runs END) AS odi_runs,
       SUM(CASE WHEN c.format = 'T20I' THEN c.runs END) AS t20i_runs,
       -- combined average = total runs / total dismissals (dismissals = runs / avg)
       ROUND(SUM(CASE WHEN c.batting_avg > 0 THEN c.runs END) * 1.0
             / SUM(CASE WHEN c.batting_avg > 0 THEN c.runs * 1.0 / c.batting_avg END), 2)
             AS overall_batting_avg
FROM player_career_stats c
JOIN players p ON p.player_id = c.player_id
WHERE c.player_id IN (SELECT player_id
                      FROM player_career_stats
                      GROUP BY player_id
                      HAVING COUNT(DISTINCT format) >= 2)
GROUP BY p.player_id, p.full_name
ORDER BY overall_batting_avg DESC; 

100 rows

,full_name,test_runs,odi_runs,t20i_runs,overall_batting_avg
0,Johan Venter,NaN,7585.0,2190.0,48.39
1,Usman Abbasi,6887.0,10655.0,1676.0,47.92
2,Marlon Maynard,NaN,9923.0,855.0,47.91
3,Freddie Pemberton,2150.0,7221.0,2200.0,47.04
4,Liam Dawson,NaN,6013.0,2758.0,45.21
5,George Pemberton,NaN,8190.0,2430.0,44.44
6,Freddie Whitmore,9217.0,2800.0,1053.0,44.31
7,Zeeshan Rana,6798.0,2371.0,1576.0,42.98
8,Freddie Hartley,4023.0,8576.0,1293.0,42.10
9,Kabir Reddy,1507.0,6021.0,NaN,41.83


In [5]:
df = show("q12")

Q12 [Intermediate] Home vs away performance by team
----------------------------------------------------------------------
WITH team_matches AS (
    SELECT team1_id AS team_id, match_id, venue_id, winner_id FROM matches
    WHERE state = 'Complete'
    UNION ALL
    SELECT team2_id, match_id, venue_id, winner_id FROM matches
    WHERE state = 'Complete'
)
SELECT t.team_name,
       SUM(CASE WHEN v.country = t.country THEN 1 ELSE 0 END) AS home_played,
       SUM(CASE WHEN v.country = t.country AND tm.winner_id = t.team_id THEN 1 ELSE 0 END) AS home_wins,
       SUM(CASE WHEN v.country <> t.country THEN 1 ELSE 0 END) AS away_played,
       SUM(CASE WHEN v.country <> t.country AND tm.winner_id = t.team_id THEN 1 ELSE 0 END) AS away_wins
FROM team_matches tm
JOIN teams t  ON t.team_id  = tm.team_id
JOIN venues v ON v.venue_id = tm.venue_id
WHERE t.country IS NOT NULL AND v.country IS NOT NULL
GROUP BY t.team_id, t.team_name
ORDER BY home_wins + away_wins DESC; 

8 rows


,team_name,home_played,home_wins,away_played,away_wins
0,India,43,27,67,41
1,Australia,46,30,63,32
2,England,40,23,68,36
3,Pakistan,37,16,73,38
4,New Zealand,35,19,76,35
5,South Africa,44,22,65,29
6,Sri Lanka,40,20,66,26
7,West Indies,40,14,65,24


In [6]:
df = show("q13")

Q13 [Intermediate] Batting partnerships of 100+ (consecutive batters)
----------------------------------------------------------------------
SELECT pa.full_name AS batter_1,
       pb.full_name AS batter_2,
       a.runs + b.runs AS partnership_runs,
       a.innings_no,
       m.match_desc,
       m.match_date
FROM batting_performances a
JOIN batting_performances b
  ON  b.match_id = a.match_id
  AND b.innings_no = a.innings_no
  AND b.batting_position = a.batting_position + 1
JOIN players pa ON pa.player_id = a.player_id
JOIN players pb ON pb.player_id = b.player_id
JOIN matches m  ON m.match_id = a.match_id
WHERE a.runs + b.runs >= 100
ORDER BY partnership_runs DESC, m.match_date DESC; 

945 rows


,batter_1,batter_2,partnership_runs,innings_no,match_desc,match_date
0,Kabir Patel,Vikram Mehta,345,1,11th ODI,2023-06-07
1,Pieter Venter,Johan Pretorius,337,3,8th Test,2022-06-25
2,Kamran Bukhari,Zeeshan Rana,304,2,15th ODI,2022-12-17
3,Rohan Reddy,Arjun Reddy,303,2,6th ODI,2022-02-28
4,Kabir Patel,Vikram Mehta,302,1,19th ODI,2025-07-31
5,Kabir Patel,Vikram Mehta,299,2,1st ODI,2020-01-12
6,Kabir Patel,Vikram Mehta,296,1,10th ODI,2025-03-16
7,Harry Stanton,Riley Fraser,287,4,10th Test,2026-08-29
8,Kabir Patel,Vikram Mehta,282,2,27th ODI,2025-11-19
9,Jack Walker,Liam Stanton,280,1,12th ODI,2023-08-24


In [7]:
df = show("q14")

Q14 [Intermediate] Bowling at venues (3+ matches, 4+ overs each)
----------------------------------------------------------------------
SELECT p.full_name,
       v.venue_name,
       COUNT(DISTINCT b.match_id)   AS matches_played,
       ROUND(AVG(b.economy_rate), 2) AS avg_economy,
       SUM(b.wickets)                AS total_wickets
FROM bowling_performances b
JOIN matches m ON m.match_id = b.match_id
JOIN venues v  ON v.venue_id = m.venue_id
JOIN players p ON p.player_id = b.player_id
WHERE b.overs >= 4
GROUP BY b.player_id, m.venue_id
HAVING COUNT(DISTINCT b.match_id) >= 3
ORDER BY avg_economy ASC, total_wickets DESC; 

590 rows


,full_name,venue_name,matches_played,avg_economy,total_wickets
0,Tane McRae,Eden Gardens,3,2.47,8
1,Dinesh Rajapaksa,Lord's,3,2.76,6
2,Salman Abbasi,Lord's,7,2.77,11
3,Cameron Walker,The Oval,3,2.77,6
4,Dinesh Gunaratne,Lord's,3,2.88,14
5,Salman Abbasi,Wanderers Stadium,3,2.96,4
6,Tane McRae,Galle International Stadium,9,3.02,15
7,Archie Pemberton,Gaddafi Stadium,8,3.05,19
8,Andre Gittens,Lord's,7,3.06,12
9,Archie Calloway,Sydney Cricket Ground,3,3.09,6


In [8]:
df = show("q15")

Q15 [Intermediate] Batters in close matches
----------------------------------------------------------------------
-- close = won by < 50 runs OR by < 5 wickets
WITH close_matches AS (
    SELECT match_id, winner_id
    FROM matches
    WHERE (victory_type = 'runs'    AND victory_margin < 50)
       OR (victory_type = 'wickets' AND victory_margin < 5)
)
SELECT p.full_name,
       ROUND(AVG(b.runs), 2) AS avg_runs,
       COUNT(DISTINCT b.match_id) AS close_matches_played,
       COUNT(DISTINCT CASE WHEN cm.winner_id = b.team_id THEN b.match_id END) AS close_matches_won
FROM batting_performances b
JOIN close_matches cm ON cm.match_id = b.match_id
JOIN players p ON p.player_id = b.player_id
GROUP BY b.player_id, p.full_name
ORDER BY avg_runs DESC; 

87 rows


,full_name,avg_runs,close_matches_played,close_matches_won
0,Kabir Patel,44.33,72,45
1,Kabir Reddy,43.16,72,45
2,Liam Dawson,40.77,64,38
3,Freddie Whitmore,38.73,66,35
4,Salman Siddiqui,38.24,72,38
5,Kamran Bukhari,37.82,72,38
6,Pieter Venter,36.92,67,28
7,Vikram Mehta,35.67,72,45
8,Kamran Abbasi,35.37,72,38
9,Liam Walker,34.32,64,38


In [9]:
df = show("q16")

Q16 [Intermediate] Yearly batting trend since 2020 (5+ matches per year)
----------------------------------------------------------------------
SELECT p.full_name,
       substr(m.match_date, 1, 4) AS year,
       COUNT(DISTINCT b.match_id) AS matches_played,
       ROUND(SUM(b.runs) * 1.0 / COUNT(DISTINCT b.match_id), 2) AS avg_runs_per_match,
       ROUND(AVG(b.strike_rate), 2) AS avg_strike_rate
FROM batting_performances b
JOIN matches m ON m.match_id = b.match_id
JOIN players p ON p.player_id = b.player_id
WHERE m.match_date >= '2020-01-01'
GROUP BY b.player_id, p.full_name, year
HAVING COUNT(DISTINCT b.match_id) >= 5
ORDER BY p.full_name, year; 

608 rows


,full_name,year,matches_played,avg_runs_per_match,avg_strike_rate
0,Aditya Nair,2020,12,2.92,48.22
1,Aditya Nair,2021,10,4.20,65.27
2,Aditya Nair,2022,8,4.25,95.89
3,Aditya Nair,2023,16,3.88,97.68
4,Aditya Nair,2024,12,5.33,64.21
5,Aditya Nair,2025,11,2.91,57.15
6,Aditya Nair,2026,5,3.80,66.50
7,Aditya Verma,2020,14,11.86,72.48
8,Aditya Verma,2021,9,13.67,67.24
9,Aditya Verma,2022,10,8.60,110.85
